# NB02 — Candidate Ranking and Experimental Prioritisation

Rank MAI-positive ENIGMA isolates by experimental tractability. Composite score (60 pts max):
- Pathway completeness (4/4 > 3/4 > ...) — 40 pts
- Genus tractability for genetic manipulation — 20 pts

Note: CheckM genome quality and documented metal tolerance phenotype were originally planned components (30 pts and 10 pts respectively) but are not exposed through ENIGMA Genome Depot browser tables. Genome quality and culture phenotype must be verified externally before ordering cultures.

In [1]:
import sys, os, re, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
warnings.filterwarnings("ignore")

NOTEBOOK_DIR = Path().resolve()
PROJECT_DIR  = NOTEBOOK_DIR.parent
DATA_DIR     = PROJECT_DIR / "data"
FIG_DIR      = PROJECT_DIR / "figures"
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

print("Setup complete.")

Setup complete.


## Section 1 — Load MAI hits and QC

In [2]:
mai_hits_path = DATA_DIR / "enigma_mai_hits.csv"

if mai_hits_path.exists():
    mai_df = pd.read_csv(mai_hits_path)
    print(f"Loaded {len(mai_df)} MAI-positive genomes from {mai_hits_path}")
    print(mai_df.head())
else:
    print(f"Warning: {mai_hits_path} not found. Run NB01 first.")
    mai_df = pd.DataFrame()

Loaded 163 MAI-positive genomes from /home/hmacgregor/BERIL-research-observatory/projects/enigma_isolate_mycothiol_isomerase/data/enigma_mai_hits.csv
   genome_id                             organism_name  mshA  mshB  mshC  mai  \
0        243      Paenarthrobacter aurescens FW305-123     1     1     1    1   
1        463     Environmental isolate FW306-2-2C-D06B     1     1     1    1   
2       4190  Environmental isolate SSO-M6-C2-PAR-NZ99     1     0     0    1   
3       3704       Ensifer adhaerens EB106-05-01-XG146     1     0     0    1   
4       1160             Environmental isolate MPR-R1A     1     1     1    1   

   pathway_completeness  
0                   1.0  
1                   1.0  
2                   0.5  
3                   0.5  
4                   1.0  


In [3]:
# Extract genus from organism_name (first word); no checkm available via browser tables
if len(mai_df) > 0:
    mai_df['genus'] = mai_df['organism_name'].str.split().str[0]
    print(f"Genus distribution ({mai_df['genus'].nunique()} unique genera):")
    print(mai_df['genus'].value_counts().head(10))
else:
    print("No MAI candidates loaded.")

Genus distribution (17 unique genera):
genus
Environmental        77
Streptomyces         30
Rhizobium            18
Arthrobacter          7
Rhodococcus           7
Ensifer               4
Nocardioides          4
Actinobacteria        4
Promicromonospora     3
Paenarthrobacter      2
Name: count, dtype: int64


## Section 2 — Score and rank

In [4]:
tractable_genera = {
    'Mycobacterium': 3,
    'Corynebacterium': 3,
    'Streptomyces': 3,
    'Rhodococcus': 2,
    'Arthrobacter': 2,
    'Paenarthrobacter': 2,
    'Microbacterium': 2,
    'Nocardia': 2,
    'Brevibacterium': 1,
    'Geodermatophilus': 1
}

if len(mai_df) > 0:
    mai_df['genus_tractability'] = mai_df['genus'].map(tractable_genera).fillna(0).astype(int)
    print(f"Mapped genus tractability scores")
    print(mai_df[['organism_name', 'genus', 'genus_tractability']].head())

Mapped genus tractability scores
                              organism_name             genus  \
0      Paenarthrobacter aurescens FW305-123  Paenarthrobacter   
1     Environmental isolate FW306-2-2C-D06B     Environmental   
2  Environmental isolate SSO-M6-C2-PAR-NZ99     Environmental   
3       Ensifer adhaerens EB106-05-01-XG146           Ensifer   
4             Environmental isolate MPR-R1A     Environmental   

   genus_tractability  
0                   2  
1                   0  
2                   0  
3                   0  
4                   0  


In [5]:
if len(mai_df) > 0:
    mai_df['score_pathway'] = mai_df['pathway_completeness'] * 40
    mai_df['score_genus'] = (mai_df['genus_tractability'] / 3.0) * 20
    mai_df['score_total'] = mai_df['score_pathway'] + mai_df['score_genus']

    mai_df = mai_df.sort_values('score_total', ascending=False).reset_index(drop=True)
    mai_df['rank'] = range(1, len(mai_df) + 1)

    print(f"Composite scores for {len(mai_df)} candidates (pathway 0-40, genus 0-20):")
    print(mai_df['score_total'].describe())

Composite scores for 163 candidates (pathway 0-40, genus 0-20):
count    163.000000
mean      41.881391
std       12.483387
min       20.000000
25%       40.000000
50%       40.000000
75%       53.333333
max       60.000000
Name: score_total, dtype: float64


In [6]:
if len(mai_df) > 0:
    shortlist = mai_df.head(20).copy()
    
    display_cols = [
        'rank', 'organism_name', 'genus', 'phylum',
        'mshA', 'mshB', 'mshC', 'mai', 'pathway_completeness',
        'checkm_completeness', 'genus_tractability', 'score_total'
    ]
    available_cols = [c for c in display_cols if c in shortlist.columns]
    
    print(f"\nTop 20 candidates ranked by experimental tractability:")
    print(shortlist[available_cols].to_string())
    
    shortlist.to_csv(DATA_DIR / "enigma_candidate_shortlist.csv", index=False)
    print(f"\nSaved candidate shortlist to {DATA_DIR / 'enigma_candidate_shortlist.csv'}")
else:
    print("No candidates to rank.")


Top 20 candidates ranked by experimental tractability:
    rank                                             organism_name         genus  mshA  mshB  mshC  mai  pathway_completeness  genus_tractability  score_total
0      1  Streptomyces mirabilis YR139 : EW58DRAFT_scaffold00001.1  Streptomyces     1     1     1    1                   1.0                   3         60.0
1      2                    Streptomyces sp. BK215 : Ga0307669_101  Streptomyces     1     1     1    1                   1.0                   3         60.0
2      3                                    Streptomyces sp. BK239  Streptomyces     1     1     1    1                   1.0                   3         60.0
3      4                    Streptomyces sp. BK141 : Ga0307717_101  Streptomyces     1     1     1    1                   1.0                   3         60.0
4      5                                    Streptomyces sp. cf124  Streptomyces     1     1     1    1                   1.0                   3    

## Section 3 — Experimental recommendations

In [7]:
recommendations = '''
EXPERIMENTAL RECOMMENDATIONS FOR TOP-RANKED CANDIDATES
===============================================================================

1. GENOME CONFIRMATION
   - BLAST MAI protein sequence against each candidate's assembly
   - Confirm synteny with mshC within 5 kb window
   - Verify absence of frameshift mutations or premature stop codons

2. GROWTH & METAL TOLERANCE ASSAY
   - Determine minimum inhibitory concentration (MIC) for Cu²⁺, Cd²⁺, Zn²⁺
   - Compare metal tolerance: wild-type vs. mshC knockout vs. mai knockout
   - Use liquid minimal medium + metal salt gradient
   - Measure OD600 at 48 h; report MIC as lowest metal concentration inhibiting growth

3. BIOCHEMICAL VALIDATION
   - Heterologous expression and purification of MAI protein (N/C-His tagged)
   - Enzymatic assay: monitor malonylpyruvate → malonate + pyruvate conversion
     * Substrate: DL-malonylpyruvate (Sigma M0252 or equiv.)
     * Cofactor: L-mycothiol or L-ergothioneine
     * Assay: HPLC or spectrophotometric (λ290 nm for malonate)
   - Test MAI kinetics in presence of metal-bound mycothiol (MSH-Cu²⁺, MSH-Zn²⁺)

4. POSITIVE CONTROLS
   - Mycobacterium smegmatis mc²155 (ATCC 700084): reference MAI-positive Actinobacterium
   - Literature precedent: Newton et al. (2008) Biochemistry 47:2268-2281
     DOI: 10.1021/bi701999g

5. EXTERNAL COLLABORATION & DATA ACCESS
   - Jen Pett-Ridge (LLNL): canonical ENIGMA isolate list & original metal contamination phenotypes
   - Gwyneth Terry (LLNL): Seaborg project database & API key for historical characterization data
   - Campus contact: https://enigma.lbl.gov

'''

print(recommendations)


EXPERIMENTAL RECOMMENDATIONS FOR TOP-RANKED CANDIDATES

1. GENOME CONFIRMATION
   - BLAST MAI protein sequence against each candidate's assembly
   - Confirm synteny with mshC within 5 kb window
   - Verify absence of frameshift mutations or premature stop codons

2. GROWTH & METAL TOLERANCE ASSAY
   - Determine minimum inhibitory concentration (MIC) for Cu²⁺, Cd²⁺, Zn²⁺
   - Compare metal tolerance: wild-type vs. mshC knockout vs. mai knockout
   - Use liquid minimal medium + metal salt gradient
   - Measure OD600 at 48 h; report MIC as lowest metal concentration inhibiting growth

3. BIOCHEMICAL VALIDATION
   - Heterologous expression and purification of MAI protein (N/C-His tagged)
   - Enzymatic assay: monitor malonylpyruvate → malonate + pyruvate conversion
     * Substrate: DL-malonylpyruvate (Sigma M0252 or equiv.)
     * Cofactor: L-mycothiol or L-ergothioneine
     * Assay: HPLC or spectrophotometric (λ290 nm for malonate)
   - Test MAI kinetics in presence of metal-bound myc

## Section 4 — Summary scatter plot

In [8]:
if len(mai_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 7))

    rng = np.random.default_rng(42)
    jitter = rng.uniform(-0.02, 0.02, len(mai_df))

    scatter = ax.scatter(
        mai_df['pathway_completeness'] + jitter,
        mai_df['score_total'],
        c=mai_df['genus_tractability'],
        cmap='YlOrRd',
        s=80,
        alpha=0.7,
        edgecolors='black',
        linewidth=0.4
    )

    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Genus Tractability (0-3)', fontsize=11)

    top5 = mai_df.head(5)
    for _, row in top5.iterrows():
        ax.annotate(
            row['organism_name'].split()[0],
            (row['pathway_completeness'] + 0.01, row['score_total']),
            fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.35),
            xytext=(5, 4), textcoords='offset points'
        )

    ax.set_xlabel('Pathway Completeness (fraction of 4 genes)', fontsize=12)
    ax.set_ylabel('Composite Score (pathway + genus tractability)', fontsize=12)
    ax.set_title('MAI-Positive Candidates: Pathway Integrity vs. Experimental Score', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(FIG_DIR / "nb02_candidate_scatter.png", dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved scatter plot to {FIG_DIR / 'nb02_candidate_scatter.png'}")
else:
    print("No MAI candidates to plot.")

Saved scatter plot to /home/hmacgregor/BERIL-research-observatory/projects/enigma_isolate_mycothiol_isomerase/figures/nb02_candidate_scatter.png


## Summary

In [9]:
if len(mai_df) > 0:
    print("\n" + "="*80)
    print("CANDIDATE RANKING COMPLETE")
    print("="*80)
    print(f"\nTotal MAI-positive candidates: {len(mai_df)} genome records ({mai_df['organism_name'].nunique()} unique strains)")
    print(f"Shortlist (top 20): {len(shortlist)}")
    print(f"\nOutput files:")
    print(f"  - {DATA_DIR / 'enigma_candidate_shortlist.csv'}")
    print(f"  - {FIG_DIR / 'nb02_candidate_scatter.png'}")
    print(f"\nNext steps:")
    print(f"  1. Review shortlist for feasibility and existing literature")
    print(f"  2. Contact Jen Pett-Ridge for ENIGMA isolate culture requests")
    print(f"  3. Begin genome confirmation (BLAST MAI + synteny check)")
else:
    print("No candidates available. Ensure NB01 completed successfully.")


CANDIDATE RANKING COMPLETE

Total MAI-positive candidates: 163 genome records (136 unique strains)
Shortlist (top 20): 20

Output files:
  - /home/hmacgregor/BERIL-research-observatory/projects/enigma_isolate_mycothiol_isomerase/data/enigma_candidate_shortlist.csv
  - /home/hmacgregor/BERIL-research-observatory/projects/enigma_isolate_mycothiol_isomerase/figures/nb02_candidate_scatter.png

Next steps:
  1. Review shortlist for feasibility and existing literature
  2. Contact Jen Pett-Ridge for ENIGMA isolate culture requests
  3. Begin genome confirmation (BLAST MAI + synteny check)
